In [ ]:
import torch
from torch.utils.data import DataLoader

from timepoint import TimePointModel
from losses import TimePointOverallLoss
from data_loader import NPZLoader
from utils import collate_fn  

device = torch.device("cpu")  

In [ ]:
def collate_fn(batch):
    return {
        "x": torch.stack([b["x"] for b in batch]),
        "x_w": torch.stack([b["x_w"] for b in batch]),
        "kp": torch.stack([b["kp"] for b in batch]),
        "kp_w": torch.stack([b["kp_w"] for b in batch]),
        "match_mask": torch.stack([b["match_mask"] for b in batch])
    }

In [ ]:
train_dataset = NPZLoader(
    "../real_data",   # ← zmień ścieżkę
    use_signal="abp"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,          # ⚠️ small (match_mask!)
    shuffle=True,
    collate_fn=collate_fn
)

In [ ]:
model = TimePointModel()

model.load_state_dict(torch.load("../models/timepoint_synthetic.pth", map_location=device))

model = model.to(device)

# ? freeze encoder?

criterion = TimePointOverallLoss(
    mp=1.0,
    mn=0.1,
    lambda_desc=1.0
)

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4   
)

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()

    total_loss = 0

    for batch in train_loader:

        # --- data ---
        x = batch["x"].to(device).unsqueeze(1)        # [B,1,L]
        x_w = batch["x_w"].to(device).unsqueeze(1)

        kp = batch["kp"].to(device)
        kp_w = batch["kp_w"].to(device)

        match_mask = batch["match_mask"].to(device)

        # forward
        S, D = model(x)
        S_w, D_w = model(x_w)

        # loss 
        loss, loss_dict = criterion(
            S, kp,
            S_w, kp_w,
            D, D_w,
            match_mask
        )

        # backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"[Epoch {epoch+1}] Loss: {avg_loss:.4f}")

In [ ]:
torch.save(model.state_dict(), "../models/timepoint_finetuned.pth")